In [1]:
!nvidia-smi

Thu Apr 23 20:01:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/sanjana231100/hr-llm-alignment.git
%cd hr-llm-alignment

Cloning into 'hr-llm-alignment'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 30 (delta 4), reused 30 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 6.63 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/kaggle/working/hr-llm-alignment


In [3]:
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 datasets accelerate bitsandbytes wandb rouge-score bert-score evaluate pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 req

In [4]:
!pip install -q transformers peft==0.10.0 trl==0.8.6 datasets accelerate bitsandbytes wandb rouge-score bert-score evaluate pyyaml

In [5]:
import transformers, peft, trl, datasets, wandb, bitsandbytes
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("all imports OK")

transformers: 4.40.0
peft: 0.10.0
trl: 0.8.6
all imports OK


In [6]:
import wandb
import os

os.environ["WANDB_API_KEY"] = "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"
wandb.login(key=os.environ["WANDB_API_KEY"])
print("W&B logged in")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sanju-231100 (sanju-231100-santa-clara-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logged in


In [7]:
import sys
sys.path.append('/kaggle/working/hr-llm-alignment')

import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from src.sft.dataset import load_hr_sft_dataset

# load config
with open('configs/sft_config.yaml') as f:
    cfg = yaml.safe_load(f)

# test dataset loads
print("Testing dataset...")
dataset = load_hr_sft_dataset(cfg)
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample:")
print(dataset[0]["text"][:300])

Testing dataset...
Loading syncora HR policies dataset...


README.md: 0.00B [00:00, ?B/s]

dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Syncora size: 644


...

Career%20QA%20Dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1620 [00:00<?, ? examples/s]

Career guidance size: 1620


Map:   0%|          | 0/1620 [00:00<?, ? examples/s]


Combined dataset size: 2908
Dataset ready: 2908 examples

Sample:


Human: What are the educational requirements for an Insurance Underwriter?

Assistant: The educational requirements generally include a high school diploma, with a preference for a bachelor's degree in business, finance, or insurance. Many employers also require or prefer certification in underwri


In [8]:
print("Loading Mistral-7B with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print("Model loaded. Applying LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# check VRAM usage
print(f"\nGPU 0 memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"GPU 1 memory: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")

Loading Mistral-7B with 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded. Applying LoRA...
trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.18788152850204565

GPU 0 memory: 2.23 GB
GPU 1 memory: 3.01 GB


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/outputs/sft",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    evaluation_strategy="no",
    report_to="wandb",
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    run_name="sft-mistral-7b-hr",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

print("Starting SFT training...")
trainer.train()

print("Training complete. Saving model...")
trainer.save_model("/kaggle/working/outputs/sft")
tokenizer.save_pretrained("/kaggle/working/outputs/sft")
print("Model saved.")

2026-04-23 20:03:59.124118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776974639.524095      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776974639.670101      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776974640.594013      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776974640.594041      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776974640.594044      23 computation_placer.cc:177] computation placer alr

Map:   0%|          | 0/2908 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


Starting SFT training...


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/hr-llm-alignment/wandb/run-20260423_200426-ghfwjzas
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sft-mistral-7b-hr
wandb: ⭐️ View project at https://wandb.ai/sanju-231100-santa-clara-university/huggingface
wandb: 🚀 View run at https://wandb.ai/sanju-231100-santa-clara-university/huggingface/runs/ghfwjzas
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: T

Step,Training Loss
10,2.361500
20,0.908200
30,0.758300
40,0.583000
50,0.485400
60,0.560800
70,0.499100
80,0.545600
90,0.483600
100,0.563900


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a n

Training complete. Saving model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model saved.


In [10]:
import os

trainer.save_model("/kaggle/working/sft-mistral-7b-hr")
tokenizer.save_pretrained("/kaggle/working/sft-mistral-7b-hr")

print("Model saved. Files:")
for f in os.listdir("/kaggle/working/sft-mistral-7b-hr"):
    print(f)

Model saved. Files:
adapter_model.safetensors
README.md
tokenizer_config.json
adapter_config.json
tokenizer.model
tokenizer.json
training_args.bin
special_tokens_map.json
